In [ ]:
import numpy as np
import scipy.linalg
import math
from itertools import product
import matplotlib.pyplot as plt
from scipy.linalg import expm
from sub_function_1D import *

%reload_ext autoreload
%autoreload 2

### Parameter Settings

In [ ]:
# Time step size
tau = 1
# Number of time steps
Total_steps = 500
# Time
T = Total_steps*tau
# Number of spatial grids
num_grid = 44
# Spatial grid spacing
delta_x = 1
# Scale normalization coefficient
Lambda = 10**0
# Density
density = 1
# Permittivity
epsilon_0 = 1
# Mass
mass = 100
# Charge
q = -1
# Number of variables
N = 2*num_grid
# Upper limit of total particle number
m = 2
# Hamiltonian matrix size
M = math.comb(m+N, m)
# Number of x qubits
n_x = math.floor(np.log2(M))+1
# Number of ancilla qubits needed for U
n_a = 1
eta=1
T_real=Total_steps*tau/(192*eta*(m/2)**(5/2))

k = 2*np.pi/num_grid
x_min = -num_grid/2
x_max = num_grid/2

### Initial Variable Preparation
$
\bold{x} = \begin{bmatrix}
            u(0,0) \\
            u(\Delta x,0) \\
            u(2\Delta x,0) \\
            \vdots \\
            E(0,0) \\
            E(\Delta x,0) \\
            E(2\Delta x,0) \\
            \vdots 
\end{bmatrix}
$

In [ ]:
r = np.linspace(x_min, x_max, num_grid)
# u list
u = np.zeros((num_grid,Total_steps+1))
r = np.linspace(x_min, x_max, num_grid)
# u list
u = np.zeros((num_grid,Total_steps+1))
u[:,0] = np.sin(-k*r)
# E list
E = np.zeros((num_grid,Total_steps+1))
# Variable list
x = np.concatenate((u[:,0], E[:,0]))
normalize_matrix(x)

### Initial State Preparation
$
|\psi(\bold{x},0)\rangle = \begin{bmatrix}
                    \psi_{m=0} \\
                    cu(0,0) \\
                    cu(\Delta x,0) \\
                    cu(2\Delta x,0) \\
                    \vdots \\
                    cE(0,0) \\
                    cE(\Delta x,0) \\
                    cE(2\Delta x,0) \\
                    \vdots \\
                    \psi_{m=2}
\end{bmatrix},
\|\psi(\bold{x},0)\| = constant
$

### KvN-expm Hamiltonian Simulation Time Evolution
Sequential time evolution with infinitesimal time step $\tau$  
$|\psi(\bold{x},t+1) \rangle = \exp(-i\frac{H}{\alpha}\tau)|\psi(\bold{x},t) \rangle$ 

In [ ]:
H = Hamiltonian_matrix(delta_x,Lambda,density,epsilon_0,mass,q,m,num_grid)
H, alpha = normalize_matrix(H)

x = np.concatenate((u[:,0], E[:,0]))
#psi = np.zeros((2**(math.floor(np.log2(M))+1),Total_steps+1))
psi = np.zeros((M,Total_steps+1))
psi[:,0], norm_psi = state_preparation_expm(N,M,x,Lambda)

exp_iHtau = expm(-1j*H*tau)

psi_now = psi[:,0]

for t in range(1,Total_steps+1):
    psi[:,t] = exp_iHtau @ psi_now
    norm = np.linalg.norm(psi[:,t])
    psi[:,t] = psi[:,t] / norm
    data = psi[:,t] * norm_psi / Lambda * np.pi**(2*num_grid/4) / 2**(1/2)
    for i in range(num_grid):
        u[i,t] = data[i+1]
        E[i,t] = data[i+num_grid+1]

    y = np.concatenate((u[:,t], E[:,t]))
    psi_now, norm_psi = state_preparation_expm(2*num_grid,M,y,Lambda)
    print("steps={}".format(t))

In [ ]:
# Save in binary format
filename = 'output/CaseBC/1DAdvectionTest_expm_u_numgrid_{}_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(num_grid,n_x, delta_x, T, tau, m)
np.save(filename, u)
filename = 'output/CaseBC/1DAdvectionTest_expm_E_numgrid_{}_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(num_grid,n_x, delta_x, T, tau, m)
np.save(filename, E)